# Clasificación de ventanas por magnitud de flujo bentónico

Este notebook replica la metodología de clasificación no supervisada del notebook 5, pero aplicada a la variable de flujo bentónico (`flux_O2`) en lugar de al estado de oleaje. El objetivo es evaluar si las ventanas se pueden agrupar de forma natural en niveles de flujo (bajo, medio, alto) y cuántos clusters soportan los datos.

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/TFM_NoeliaGarciaGarcia/Pipeline'
else:
    BASE_PATH = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"BASE_PATH: {BASE_PATH}")

# CARGA DE DATOS

In [ ]:
base_path = os.path.join(BASE_PATH, "DATA", "PROCESSED")

df_wave = pd.read_csv(os.path.join(base_path, "df_wave.csv"))
df_features = pd.read_csv(os.path.join(base_path, "df_features.csv"))

df = pd.merge(
    df_wave,
    df_features,
    on="window_id",
    how="inner"
)

print(f"Ventanas cargadas: {len(df)}")
print(f"Columnas disponibles: {df.columns.tolist()}")
df[["window_id", "flux_O2", "flux_smooth", "mean_O2", "mean_vz"]].head(10)

# EXPLORACIÓN PREVIA DEL FLUJO

In [ ]:
# Distribución del flujo
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["flux_O2"].dropna(), bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("flux_O2 (mmol/m²/día)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución del flujo bentónico")
axes[0].axvline(0, color="red", linestyle="--", alpha=0.5)
axes[0].grid(True, alpha=0.3)

# Valor absoluto del flujo
df["abs_flux_O2"] = df["flux_O2"].abs()
axes[1].hist(df["abs_flux_O2"].dropna(), bins=50, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("|flux_O2| (mmol/m²/día)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución del valor absoluto del flujo")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(df["flux_O2"].describe())

# SELECCIÓN DE VARIABLES PARA CLUSTERING

Se utiliza el flujo y variables directamente relacionadas con él para la clasificación. Se usa el valor absoluto del flujo como variable principal, ya que lo que interesa es la **magnitud** del intercambio, no su signo.

In [ ]:
# Variable derivada: magnitud del flujo
df["abs_flux_O2"] = df["flux_O2"].abs()

# Features para el clustering basado en flujo
features = [
    "abs_flux_O2",
]

print(f"Variables seleccionadas: {features}")
print(f"\nEstadísticas:")
display(df[features].describe())

In [ ]:
# Preparar la matriz X
X = df[features].copy()

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_imputed = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imputed)

# EVALUACIÓN DE NÚMERO DE CLUSTERS

In [ ]:
k_values = range(2, 9)

inertias = []
silhouettes = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

df_k_eval = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertias,
    "silhouette": silhouettes
})

display(df_k_eval)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df_k_eval["k"], df_k_eval["inertia"], marker="o")
axes[0].set_xlabel("Número de clusters")
axes[0].set_ylabel("Inercia")
axes[0].set_title("Método del codo")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_k_eval["k"], df_k_eval["silhouette"], marker="o", color="orange")
axes[1].set_xlabel("Número de clusters")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Evaluación silhouette")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# APLICACIÓN K-MEANS CON k=3 (flujo bajo, medio, alto)

In [ ]:
k_flujo = 3

kmeans = KMeans(
    n_clusters=k_flujo,
    random_state=42,
    n_init=20
)

df["cluster_flujo"] = kmeans.fit_predict(X_scaled)

print(f"Silhouette score con k={k_flujo}: {silhouette_score(X_scaled, df['cluster_flujo']):.4f}")

In [ ]:
# Ordenar clusters por magnitud de flujo (de menor a mayor abs_flux_O2)
orden_clusters = (
    df
    .groupby("cluster_flujo")["abs_flux_O2"]
    .mean()
    .sort_values()
    .index
    .tolist()
)

mapa_clusters = {
    cluster_original: nuevo_id
    for nuevo_id, cluster_original in enumerate(orden_clusters)
}

df["flujo_clase"] = df["cluster_flujo"].map(mapa_clusters)

nombres_clase = {
    0: "flujo_bajo",
    1: "flujo_medio",
    2: "flujo_alto"
}

df["flujo_clase_nombre"] = df["flujo_clase"].map(nombres_clase)

print("Distribución de clases:")
print(df["flujo_clase_nombre"].value_counts().sort_index())

# RESUMEN POR CLUSTER

In [ ]:
resumen = (
    df
    .groupby(["flujo_clase", "flujo_clase_nombre"])
    .agg(
        n_ventanas=("flux_O2", "count"),
        flux_mean=("flux_O2", "mean"),
        flux_std=("flux_O2", "std"),
        flux_median=("flux_O2", "median"),
        flux_min=("flux_O2", "min"),
        flux_max=("flux_O2", "max"),
        abs_flux_mean=("abs_flux_O2", "mean"),
        abs_flux_std=("abs_flux_O2", "std"),
    )
)

display(resumen)

# VISUALIZACIÓN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot flux_O2 por clase
sns.boxplot(data=df, x="flujo_clase_nombre", y="flux_O2", ax=axes[0],
            order=["flujo_bajo", "flujo_medio", "flujo_alto"])
axes[0].set_title("Flujo bentónico por clase de flujo")
axes[0].set_xlabel("Clase de flujo")
axes[0].set_ylabel("flux_O2 (mmol/m²/día)")
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].grid(True, alpha=0.3)

# Boxplot |flux_O2| por clase
sns.boxplot(data=df, x="flujo_clase_nombre", y="abs_flux_O2", ax=axes[1],
            order=["flujo_bajo", "flujo_medio", "flujo_alto"])
axes[1].set_title("|Flujo| por clase de flujo")
axes[1].set_xlabel("Clase de flujo")
axes[1].set_ylabel("|flux_O2| (mmol/m²/día)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Histograma del flujo coloreado por clase
plt.figure(figsize=(10, 5))
for clase in ["flujo_bajo", "flujo_medio", "flujo_alto"]:
    subset = df[df["flujo_clase_nombre"] == clase]
    plt.hist(subset["flux_O2"], bins=30, alpha=0.5, label=clase, edgecolor="black")

plt.xlabel("flux_O2 (mmol/m²/día)")
plt.ylabel("Frecuencia")
plt.title("Distribución de flujo por clase")
plt.legend()
plt.axvline(0, color="red", linestyle="--", alpha=0.5)
plt.grid(True, alpha=0.3)
plt.show()

# CRUCE CON ESTADO DE OLEAJE

Se compara la clasificación por flujo con la clasificación por oleaje del notebook 5 para ver si existe correspondencia.

In [ ]:
# Cargar la clasificación original de oleaje si existe
df_classified_path = os.path.join(BASE_PATH, "DATA", "PROCESSED", "df_classified.csv")
if os.path.exists(df_classified_path):
    df_oleaje_cls = pd.read_csv(df_classified_path)[["window_id", "oleaje_clase", "oleaje_clase_nombre"]]
    df = df.merge(df_oleaje_cls, on="window_id", how="left")

    # Tabla de contingencia
    tabla = pd.crosstab(
        df["flujo_clase_nombre"],
        df["oleaje_clase_nombre"],
        margins=True
    )
    print("Tabla de contingencia: clase de flujo × clase de oleaje")
    display(tabla)
else:
    print("No se encontró df_classified.csv. Ejecuta primero el notebook 5.")

In [ ]:
# Boxplot de oleaje por clase de flujo
if "oleaje_clase_nombre" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.boxplot(data=df, x="flujo_clase_nombre", y="Oleaje", ax=axes[0],
                order=["flujo_bajo", "flujo_medio", "flujo_alto"])
    axes[0].set_title("Oleaje según clase de flujo")
    axes[0].set_xlabel("Clase de flujo")
    axes[0].set_ylabel("Oleaje")
    axes[0].grid(True, alpha=0.3)

    sns.boxplot(data=df, x="flujo_clase_nombre", y="Mod_orbital", ax=axes[1],
                order=["flujo_bajo", "flujo_medio", "flujo_alto"])
    axes[1].set_title("Velocidad orbital según clase de flujo")
    axes[1].set_xlabel("Clase de flujo")
    axes[1].set_ylabel("Mod_orbital")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# CRUCE CON FLAGS DE CALIDAD

In [ ]:
flags = ["flag_outlier_window", "flag_outlier_global", "flag_flux_positivo", "flag_vz_baja"]
flags_disponibles = [f for f in flags if f in df.columns]

if flags_disponibles:
    resumen_flags = (
        df
        .groupby("flujo_clase_nombre")[flags_disponibles]
        .mean()
        .round(3)
    )
    print("Proporción de flags por clase de flujo:")
    display(resumen_flags)
else:
    print("No se encontraron flags de calidad en el dataframe.")

# PRUEBA CON k=2 PARA COMPARAR

In [ ]:
kmeans_k2 = KMeans(n_clusters=2, random_state=42, n_init=20)
df["cluster_flujo_k2"] = kmeans_k2.fit_predict(X_scaled)

sil_k2 = silhouette_score(X_scaled, df["cluster_flujo_k2"])
sil_k3 = silhouette_score(X_scaled, df["cluster_flujo"])

print(f"Silhouette k=2: {sil_k2:.4f}")
print(f"Silhouette k=3: {sil_k3:.4f}")
print(f"\nMejor k según silhouette: {'k=2' if sil_k2 > sil_k3 else 'k=3'}")

# Resumen rápido de k=2
orden_k2 = (
    df.groupby("cluster_flujo_k2")["abs_flux_O2"].mean()
    .sort_values().index.tolist()
)
mapa_k2 = {c: i for i, c in enumerate(orden_k2)}
df["flujo_clase_k2"] = df["cluster_flujo_k2"].map(mapa_k2)
df["flujo_clase_nombre_k2"] = df["flujo_clase_k2"].map({0: "flujo_bajo", 1: "flujo_alto"})

resumen_k2 = (
    df.groupby("flujo_clase_nombre_k2")
    .agg(
        n_ventanas=("flux_O2", "count"),
        flux_mean=("flux_O2", "mean"),
        flux_std=("flux_O2", "std"),
        abs_flux_mean=("abs_flux_O2", "mean"),
    )
)
display(resumen_k2)

# CONCLUSIONES

Observar:
- ¿El silhouette score es razonable para k=3? Si es muy bajo (<0.3), la separación en 3 grupos no es clara.
- ¿Existe correspondencia entre las clases de flujo y las de oleaje?
- ¿Los flags de calidad se concentran en alguna clase de flujo?
- ¿k=2 separa mejor que k=3?